# Zürich Tram Flow — Analyse-Report

**93.9 Mio. Datenpunkte · 3 Jahre (2023–2025) · 16 Tramlinien · Zürich**

Dieser Report fasst die wichtigsten Erkenntnisse der Analyse-Phase zusammen.
Er richtet sich an alle die verstehen wollen, wo, wann und warum Trams in Zürich verspätet sind —
und was das für den Betrieb bedeutet.

## Datenbasis

| | |
|:---|:---|
| **Quelle** | VBZ Zürich — IST-Daten opentransportdata.swiss |
| **Zeitraum** | Januar 2023 – Oktober 2025 |
| **Umfang** | 93.9 Mio. Zeilen · 16 Tramlinien · 24 Features |
| **Analysebasis (lf_clean)** | canceled==False · stop_sequence>1 · ohne Linie E · ohne Nov/Dez 2025 |
| **OTP-Schwellwert** | ±120s (VBZ-Standard) |

Alle Zahlen in diesem Report basieren auf der bereinigten Datenbasis (lf_clean) — ausser wo explizit anders angegeben.

## Setup

In [ ]:
from zh_tram_flow.notebook import *
import zh_tram_flow.analytics as an

TRAIN, TEST, lf = setup_analysis("04_insights")

lf_all   = pl.concat([pl.scan_parquet(TRAIN), pl.scan_parquet(TEST)])
lf_delay = lf_all.filter(pl.col("canceled") == False)
lf_clean = (
    lf_all
    .filter(pl.col("canceled") == False)
    .filter(~((pl.col("operating_date").dt.year() == 2025) & (pl.col("operating_date").dt.month() >= 11)))
    .filter(pl.col("line_name") != "E")
    .filter(pl.col("stop_sequence") > 1)
)

## 1. Das System 



Dezember 2023: der **grösste Fahrplanwechsel der VBZ-Geschichte** — L9, L11 und L13 fundamental umgebaut. Netzweites Delay-Signal: **+0.5s**. Veränderte Linien (L11 +5.3s) und stabile Linien (L15 +5.2s) zeigen identische Verläufe. Der Umbau hat weder geholfen noch geschadet — das Problem liegt woanders.

In [ ]:
an.plot_monthly_delay_all_lines(lf_all, cfg)
show_df(an.table_delay_before_after_switch(lf_all))

**71.5% aller Halte akkumulieren Verspätung** — kein Puffer ist eingebaut. Die OTP liegt netzweit bei 87% — stabil, aber strukturell fragil. Der Grund: **71.3% aller geplanten Haltezeiten (dwell_time) betragen 0 Sekunden**. Das System hat buchstäblich keinen Spielraum für Störungen — jede Verzögerung pflanzt sich fort.

In [ ]:
an.plot_dwell_time(lf_clean, cfg)
show_df(an.table_dwell_time_by_line(lf_clean))

## 2. Wo entstehen Verspätungen?



Die Hotspots sind **periphere Aussenkorridore** — nicht die zentralen Knotenpunkte. Friedhof Enzenbühl (93.8s), Balgrist (85.2s) und Leutschenbach (82.7s) führen die Liste an. Central (48.3s) und Paradeplatz (48.2s), wo je 14–15 Linien kreuzen, liegen **unter dem Netzschnitt**. Stadtkreis 11 (68.3s, OTP 83%) ist der schlechteste Kreis — Kreis 5 der beste (49.9s, OTP 89%). **0 Überschneidung** zwischen Top-20 nach Liniendichte und Top-20 nach Delay.

In [ ]:
an.plot_stop_delay_map(lf_clean)
show_df(an.table_stop_delay_map(lf_clean))

## 3. Wann entstehen Verspätungen?



**Kein Morgenrush.** 7h liegt mit 48.9s unter dem Netzschnitt — der Peak ist um 21h (67.9s), getrieben von Abreisewellen nach Konzerten und Spielen. Donnerstag ist der schlechteste Wochentag (60.4s, P95=194s) — nicht Freitag. November ist der schlechteste Monat (2024: 72.6s), Winter überraschend die beste Jahreszeit (51.7s, OTP 88.9%).

In [ ]:
an.plot_hour_of_day(lf_clean, cfg)
show_df(an.table_hour_of_day(lf_clean))
an.plot_day_of_week(lf_clean, cfg)
show_df(an.table_day_of_week(lf_clean))

## 4. Wettereinflüsse 



**Schnee** ist der stärkste Einzeleinflussfaktor im Datensatz: **+54s Mehrdelay, OTP −10.9 Prozentpunkte**. Die geografische Trennung ist klar: Schnee trifft Höhenlagen (Kreise 10/4/12), Regen trifft Flusstäler (Kreis 5). Linien reagieren entgegengesetzt — L17 leidet unter Regen (+41.2s), L9 unter Schnee (+75.9s). Kälte (0–5°C) ist die **beste** Wetterbedingung (53.8s) — Frost schützt das Netz eher als er schadet.

In [ ]:
an.plot_district_weather_sensitivity(lf_clean, cfg)
show_df(an.table_district_weather_sensitivity(lf_clean))
an.plot_line_weather_exposure(lf_clean, cfg)
show_df(an.table_line_weather_exposure(lf_clean))

**Winter ist die beste Jahreszeit** — 51.7s (OTP 88.9%), besser als Frühling (55.6s) und Sommer (56.4s). Herbst ist die schlechteste (61.2s). Der Rückgang des Kfz-Verkehrs im Winter überwiegt den Schnee-Effekt.

## 5. Events & Feiertage



**Feiertage sind die besten Tage** — 46.3s, −9.9s gegenüber Normal, OTP 90.6% vs. 87.0%. Der Rückgang des Berufsverkehrs überwiegt jeden Eventeffekt. Grosse Events kosten +10.5s — aber fast ausschliesslich **abends zwischen 18 und 22 Uhr**. Tagsüber kein messbarer Unterschied.

In [ ]:
an.plot_events_overview(lf_clean, cfg)
show_df(an.table_events_overview(lf_clean))

**Fachmessen** (66.0s, OTP 84%) sind die schlechteste Ereigniskategorie — nicht Konzerte oder Fussball. Der schlechteste Tag im Datensatz: Berufsmesse Zürich, 21. November 2024 — **192.5s Ø Delay, OTP 54.5%**. Taylor Swift: 75.4s — weniger als halb so viel.

In [ ]:
an.plot_daily_delay_timeline(lf_clean, cfg)
show_df(an.table_daily_delay_timeline(lf_clean))

## 6. Netz — Ausbau ohne Wirkung an den richtigen Orten


Die Streckenerweiterungen gingen nach Sihlcity (K3, 55.7s) und Rehalp (K8, 63.7s) — beide gut performende Gebiete. Kreise 11 (68.3s, OTP 83%) und 12 (66.3s), die eigentlichen Problemzonen, erhielten nichts. **0 Überschneidung** zwischen Investitionsort und Problemort.

In [ ]:
an.plot_service_quality_district_map(lf_all)
show_df(an.table_service_quality_by_district(lf_all))

## Fazit & Empfehlungen

Das Zürcher Tramnetz hat strukturelle Verspätungsmuster die über drei Jahre stabil sind. Die Analyse zeigt drei klare Handlungsfelder:

| Priorität | Empfehlung | Basis |
|:---|:---|:---|
| **1 — Hoch** | Fahrplanpuffer gezielt in Aussenkorridoren einbauen — K11/K12 brauchen mehr Haltezeit-Reserven, nicht mehr Linien | F-SPAT-01, F-TARGET-03 |
| **2 — Mittel** | Abend-Kapazität prüfen (18–22h) — Events + Feierabend treffen gleichzeitig. Sonderkonzept für Donnerstag-Abend prüfen | F-TEMP-01/02, F-EVNT-03 |
| **3 — Mittel** | Netzinvestitionen auf Hotspot-Kreise ausrichten — zukünftige Erweiterungen zuerst in K11/K12 evaluieren, nicht in bereits gut performenden Gebieten | F-NET-09, F-SPAT-03 |

**Was kommt:** Ein Vorhersagemodell (LightGBM) soll auf Basis dieser Features quantifizieren, welche Faktoren wie viel zur Verspätung beitragen — und welche Halte zu welcher Zeit am stärksten gefährdet sind.